<a href="https://colab.research.google.com/github/payalbamane018-sudo/Buyer_Segmentation_and_Investment_Profiling_for_Real_Estate_Market_Intelligence/blob/main/Unified_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the dataset
df = pd.read_csv('clients.csv')

# 2. Handle missing values (Explicit & Implicit)
# Replace common missing value indicators with NaN
missing_indicators = ['', ' ', 'None', 'none', 'null', 'Null', 'NaN', 'nan', 'N/A', 'n/a', '?']
df.replace(missing_indicators, np.nan, inplace=True)

# Drop or fill missing values if present
df.dropna(subset=['client_id'], inplace=True) # Ensure key identifier is present

# 3. Normalize categorical labels
categorical_cols = [
    'client_type', 'gender', 'country', 'region',
    'acquisition_purpose', 'loan_applied', 'referral_channel'
]

for col in categorical_cols:
    if col in df.columns:
        # Standardize string formatting: strip whitespace & apply title casing
        df[col] = df[col].astype(str).str.strip().str.title()

# Convert standard abbreviations or formatting back if needed
df['gender'] = df['gender'].str.upper() # 'M', 'F'
df['country'] = df['country'].replace({'Usa': 'USA', 'Uk': 'UK'})

# 4. Standardize date formats
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], format='mixed').dt.strftime('%Y-%m-%d')

# 5. Remove duplicate client entries
# Check duplicates by client_id as well as overall content
df.drop_duplicates(subset=['client_id'], keep='first', inplace=True)
df.drop_duplicates(subset=df.columns.difference(['client_id']), keep='first', inplace=True)

# Reset index after cleaning
df.reset_index(drop=True, inplace=True)

# 6. Save cleaned dataset for Step 2
df.to_csv('clients_cleaned.csv', index=False)
print("Step 1 Complete! Cleaned dataset saved as 'clients_cleaned.csv'.")

Step 1 Complete! Cleaned dataset saved as 'clients_cleaned.csv'.


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# 1. Load the cleaned data from Step 1
df = pd.read_csv('clients_cleaned.csv')

# 2. Apply Label Encoding for binary categorical variables
label_cols = ['client_type', 'acquisition_purpose']
label_encoders = {}

for col in label_cols:
    le = LabelEncoder()
    df[f'{col}_label'] = le.fit_transform(df[col])
    label_encoders[col] = le

# 3. Apply One-Hot Encoding for nominal categorical variables
ohe_cols = ['country', 'region', 'referral_channel']
df_encoded = pd.get_dummies(df, columns=ohe_cols, prefix=ohe_cols, dtype=int)

# 4. Inspect result and save encoded dataset for Step 3
print("Encoded Dataset Shape:", df_encoded.shape)
df_encoded.to_csv('clients_encoded.csv', index=False)
print("Step 2 Complete! Encoded dataset saved as 'clients_encoded.csv'.")

Encoded Dataset Shape: (2000, 81)
Step 2 Complete! Encoded dataset saved as 'clients_encoded.csv'.


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Load dataset from Step 2
df = pd.read_csv('clients_encoded.csv')

# 2. Calculate Age from date_of_birth if not already present
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], format='mixed')
current_year = pd.Timestamp.now().year
df['age'] = current_year - df['date_of_birth'].dt.year

# 3. Select numerical features to scale
num_cols = ['age', 'satisfaction_score']

# 4. Apply StandardScaler (z-score normalization)
scaler = StandardScaler()
df[[f'{col}_scaled' for col in num_cols]] = scaler.fit_transform(df[num_cols])

# 5. Inspect scaled features and save dataset for Step 4
print(df[['age', 'age_scaled', 'satisfaction_score', 'satisfaction_score_scaled']].head())

df.to_csv('clients_scaled.csv', index=False)
print("Step 3 Complete! Scaled dataset saved as 'clients_scaled.csv'.")

   age  age_scaled  satisfaction_score  satisfaction_score_scaled
0   58    0.109635                   4                   0.687089
1   64    0.455668                   1                  -1.435740
2   67    0.628684                   4                   0.687089
3   67    0.628684                   5                   1.394698
4   50   -0.351743                   5                   1.394698
Step 3 Complete! Scaled dataset saved as 'clients_scaled.csv'.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

# 1. Load the preprocessed dataset
df = pd.read_csv('clients_scaled.csv')

# 2. Select numerical & binary encoded features for clustering
feature_cols = [
    'age_scaled',
    'satisfaction_score_scaled',
    'client_type_label',
    'acquisition_purpose_label'
]

X = df[feature_cols].values

# -------------------------------------------------------------
# Approach 1: K-Means Clustering
# -------------------------------------------------------------
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['kmeans_cluster'] = kmeans.fit_predict(X)

# -------------------------------------------------------------
# Approach 2: Hierarchical Clustering & Dendrogram
# -------------------------------------------------------------
agg_cluster = AgglomerativeClustering(n_clusters=4, metric='euclidean', linkage='ward')
df['hierarchical_cluster'] = agg_cluster.fit_predict(X)

# Generate Dendrogram to visualize nested relationships
plt.figure(figsize=(10, 5))
plt.title("Hierarchical Clustering Dendrogram")
linked = linkage(X, method='ward')
dendrogram(linked, truncate_mode='lastp', p=20, leaf_rotation=45., leaf_font_size=10., show_contracted=True)
plt.xlabel("Cluster Size")
plt.ylabel("Euclidean Distance")
plt.tight_layout()
plt.savefig('hierarchical_dendrogram.png')
plt.close()

# -------------------------------------------------------------
# Map & Validate Recommended Buyer Segments (C1 - C4)
# -------------------------------------------------------------
cluster_mapping = {
    0: 'C1 - Global Investors',
    1: 'C2 - First-Time Buyers',
    2: 'C3 - Corporate Buyers',
    3: 'C4 - Luxury Investors'
}

df['buyer_segment'] = df['kmeans_cluster'].map(cluster_mapping)

# Print Summary Verification
print("--- Cluster Distribution ---")
print(df['buyer_segment'].value_counts())

# Save clustered dataset for Step 5
df.to_csv('clients_clustered.csv', index=False)
print("\nStep 4 Complete! Clustered dataset saved as 'clients_clustered.csv' and dendrogram saved as 'hierarchical_dendrogram.png'.")

--- Cluster Distribution ---
buyer_segment
C2 - First-Time Buyers    605
C3 - Corporate Buyers     580
C4 - Luxury Investors     423
C1 - Global Investors     392
Name: count, dtype: int64

Step 4 Complete! Clustered dataset saved as 'clients_clustered.csv' and dendrogram saved as 'hierarchical_dendrogram.png'.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Load the scaled feature dataset
df = pd.read_csv('clients_scaled.csv')

# Select features used for clustering
feature_cols = [
    'age_scaled',
    'satisfaction_score_scaled',
    'client_type_label',
    'acquisition_purpose_label'
]

X = df[feature_cols].values

# 2. Compute WCSS (Inertia) and Silhouette Scores for K = 2 to 10
k_range = range(2, 11)
wcss = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X)

    wcss.append(kmeans.inertia_)
    score = silhouette_score(X, cluster_labels)
    silhouette_scores.append(score)
    print(f"Clusters (k={k}): WCSS = {kmeans.inertia_:.2f} | Silhouette Score = {score:.4f}")

# 3. Plot Elbow Method (WCSS vs K) and Silhouette Scores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Plot
ax1.plot(k_range, wcss, 'bo-', linewidth=2, markersize=8)
ax1.set_title('Elbow Method For Optimal k')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Within-Cluster Sum of Squares (WCSS)')
ax1.grid(True)

# Silhouette Score Plot
ax2.plot(k_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
ax2.set_title('Silhouette Score For Optimal k')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.grid(True)

plt.tight_layout()
plt.savefig('cluster_evaluation_metrics.png')
plt.close()

print("\nStep 5 Complete! Metrics visual saved as 'cluster_evaluation_metrics.png'.")

Clusters (k=2): WCSS = 3024.78 | Silhouette Score = 0.3149
Clusters (k=3): WCSS = 2086.69 | Silhouette Score = 0.3232
Clusters (k=4): WCSS = 1551.17 | Silhouette Score = 0.3278
Clusters (k=5): WCSS = 1363.97 | Silhouette Score = 0.3037
Clusters (k=6): WCSS = 1215.73 | Silhouette Score = 0.2963
Clusters (k=7): WCSS = 1108.82 | Silhouette Score = 0.2901
Clusters (k=8): WCSS = 1032.20 | Silhouette Score = 0.3153
Clusters (k=9): WCSS = 939.13 | Silhouette Score = 0.2959
Clusters (k=10): WCSS = 872.94 | Silhouette Score = 0.2929

Step 5 Complete! Metrics visual saved as 'cluster_evaluation_metrics.png'.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load original clients dataset to preserve raw categorical columns ('country', 'loan_applied', etc.)
df_raw = pd.read_csv('clients.csv')

# Load the encoded/scaled/clustered file if available
try:
    df_clustered = pd.read_csv('clients_clustered.csv')
except FileNotFoundError:
    df_clustered = pd.read_csv('clients_scaled.csv')

# 2. Combine cluster labels with raw data
if 'buyer_segment' in df_clustered.columns:
    df_raw['buyer_segment'] = df_clustered['buyer_segment']
elif 'kmeans_cluster' in df_clustered.columns:
    cluster_mapping = {
        0: 'C1 - Global Investors',
        1: 'C2 - First-Time Buyers',
        2: 'C3 - Corporate Buyers',
        3: 'C4 - Luxury Investors'
    }
    df_raw['buyer_segment'] = df_clustered['kmeans_cluster'].map(cluster_mapping)
else:
    # Rule-based segment assignment backup
    df_raw['buyer_segment'] = 'C1 - Global Investors'
    df_raw.loc[df_raw['client_type'] == 'Company', 'buyer_segment'] = 'C3 - Corporate Buyers'
    df_raw.loc[(df_raw['client_type'] != 'Company') & (df_raw['acquisition_purpose'] == 'Home'), 'buyer_segment'] = 'C2 - First-Time Buyers'

# 3. Compute Age feature
df_raw['date_of_birth'] = pd.to_datetime(df_raw['date_of_birth'], format='mixed')
df_raw['age'] = 2026 - df_raw['date_of_birth'].dt.year

# 4. Generate Cluster Interpretation Summary Table safely
cluster_summary = df_raw.groupby('buyer_segment').agg(
    total_clients=('client_id', 'count'),
    mean_age=('age', 'mean'),
    pct_investment=('acquisition_purpose', lambda x: (x == 'Investment').mean() * 100),
    pct_loan_applied=('loan_applied', lambda x: (x == 'Yes').mean() * 100),
    pct_company=('client_type', lambda x: (x == 'Company').mean() * 100),
    top_country=('country', lambda x: x.mode()[0]),
    mean_satisfaction=('satisfaction_score', 'mean')
).round(2)

print("=== CLUSTER INTERPRETATION SUMMARY ===")
print(cluster_summary)

# Save output table
cluster_summary.to_csv('cluster_interpretation_report.csv')

# 5. Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Visual 1: Investment Purpose
sns.countplot(data=df_raw, x='buyer_segment', hue='acquisition_purpose', ax=axes[0, 0], palette='Blues')
axes[0, 0].set_title('1. Investment Purpose by Cluster')
axes[0, 0].tick_params(axis='x', rotation=15)

# Visual 2: Geographic Distribution (Top Countries)
top_countries = df_raw['country'].value_counts().head(5).index
df_filtered_countries = df_raw[df_raw['country'].isin(top_countries)]
sns.countplot(data=df_filtered_countries, x='buyer_segment', hue='country', ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('2. Top Geographic Distribution by Cluster')
axes[0, 1].tick_params(axis='x', rotation=15)
axes[0, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Visual 3: Loan Behavior
sns.countplot(data=df_raw, x='buyer_segment', hue='loan_applied', ax=axes[1, 0], palette='Greens')
axes[1, 0].set_title('3. Loan Behavior by Cluster')
axes[1, 0].tick_params(axis='x', rotation=15)

# Visual 4: Demographics (Age)
sns.boxplot(data=df_raw, x='buyer_segment', y='age', ax=axes[1, 1], palette='Oranges')
axes[1, 1].set_title('4. Age Demographics by Cluster')
axes[1, 1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('cluster_interpretation_visuals.png')
plt.close()

print("\nStep 6 Complete! Output saved as 'cluster_interpretation_report.csv' and 'cluster_interpretation_visuals.png'.")

=== CLUSTER INTERPRETATION SUMMARY ===
                        total_clients  mean_age  pct_investment  \
buyer_segment                                                     
C1 - Global Investors             392     41.20           33.67   
C2 - First-Time Buyers            605     70.45           27.77   
C3 - Corporate Buyers             580     40.68           34.14   
C4 - Luxury Investors             423     70.52           27.66   

                        pct_loan_applied  pct_company top_country  \
buyer_segment                                                       
C1 - Global Investors              37.50         7.65         USA   
C2 - First-Time Buyers             36.03         2.31         USA   
C3 - Corporate Buyers              39.31         7.93         USA   
C4 - Luxury Investors              33.81         3.07         USA   

                        mean_satisfaction  
buyer_segment                              
C1 - Global Investors                1.44  
C2 - First-

/tmp/ipykernel_6492/678985848.py:74: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_raw, x='buyer_segment', y='age', ax=axes[1, 1], palette='Oranges')



Step 6 Complete! Output saved as 'cluster_interpretation_report.csv' and 'cluster_interpretation_visuals.png'.


In [24]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import os

# -----------------------------------------------------------------------------
# 1. Page Configuration
# -----------------------------------------------------------------------------
st.set_page_config(
    page_title="Real Estate Market Intelligence",
    page_icon="🏢",
    layout="wide"
)

st.title("🏢 Real Estate Market Intelligence & Buyer Segmentation")
st.markdown("Interactive analytics dashboard for data-driven buyer profiling and investment intelligence.")

# -----------------------------------------------------------------------------
# 2. Robust Data Loading
# -----------------------------------------------------------------------------
@st.cache_data
def load_data():
    # Check multiple possible paths for dataset
    possible_paths = [
        'clients.csv',
        'clients_clustered.csv',
        'data/clients.csv',
        'data/clients_clustered.csv',
        'data/clients_cleaned.csv'
    ]

    df = None
    for path in possible_paths:
        if os.path.exists(path):
            df = pd.read_csv(path)
            break

    if df is None:
        raise FileNotFoundError("Could not find dataset in repository root or data folder.")

    # Calculate Age
    if 'age' not in df.columns and 'date_of_birth' in df.columns:
        df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], format='mixed')
        df['age'] = 2026 - df['date_of_birth'].dt.year

    # Ensure buyer segment column exists
    if 'buyer_segment' not in df.columns:
        if 'kmeans_cluster' in df.columns:
            cluster_mapping = {
                0: 'C1 - Global Investors',
                1: 'C2 - First-Time Buyers',
                2: 'C3 - Corporate Buyers',
                3: 'C4 - Luxury Investors'
            }
            df['buyer_segment'] = df['kmeans_cluster'].map(cluster_mapping)
        else:
            def assign_segment(row):
                if row.get('client_type') == 'Company':
                    return 'C3 - Corporate Buyers'
                elif row.get('acquisition_purpose') == 'Investment':
                    if row.get('country') != 'USA':
                        return 'C1 - Global Investors'
                    else:
                        return 'C4 - Luxury Investors'
                else:
                    return 'C2 - First-Time Buyers'
            df['buyer_segment'] = df.apply(assign_segment, axis=1)

    return df

try:
    df = load_data()
except Exception as e:
    st.error(f"Error loading dataset: {e}")
    st.stop()

# -----------------------------------------------------------------------------
# 3. Sidebar Filters
# -----------------------------------------------------------------------------
st.sidebar.header("🎯 Dashboard Filters")

selected_segments = st.sidebar.multiselect(
    "Select Buyer Segments:",
    options=df['buyer_segment'].unique(),
    default=df['buyer_segment'].unique()
)

selected_countries = st.sidebar.multiselect(
    "Select Country:",
    options=df['country'].unique() if 'country' in df.columns else [],
    default=df['country'].unique() if 'country' in df.columns else []
)

# Apply Filter
filtered_df = df[df['buyer_segment'].isin(selected_segments)]
if 'country' in df.columns and selected_countries:
    filtered_df = filtered_df[filtered_df['country'].isin(selected_countries)]

# Metrics Header
st.markdown("---")
kpi1, kpi2, kpi3, kpi4 = st.columns(4)
kpi1.metric("Total Clients", f"{len(filtered_df):,}")
if 'satisfaction_score' in filtered_df.columns:
    kpi2.metric("Avg Satisfaction", f"{filtered_df['satisfaction_score'].mean():.2f} / 5.0")
if 'loan_applied' in filtered_df.columns:
    kpi3.metric("Loan Dependency Rate", f"{(filtered_df['loan_applied'] == 'Yes').mean() * 100:.1f}%")
if 'acquisition_purpose' in filtered_df.columns:
    kpi4.metric("Investment Buyers Rate", f"{(filtered_df['acquisition_purpose'] == 'Investment').mean() * 100:.1f}%")
st.markdown("---")

# -----------------------------------------------------------------------------
# 4. Tabs & Dashboard Visuals
# -----------------------------------------------------------------------------
tab1, tab2, tab3, tab4 = st.tabs([
    "📊 Buyer Segmentation Overview",
    "📈 Investor Behavior Dashboard",
    "🌍 Geographic Buyer Analysis",
    "📋 Segment Insights Panel"
])

with tab1:
    st.header("Buyer Segmentation Overview")
    col1, col2 = st.columns(2)
    with col1:
        st.subheader("Cluster Volume Distribution")
        fig_pie = px.pie(filtered_df, names='buyer_segment', hole=0.4, color_discrete_sequence=px.colors.qualitative.Set2)
        st.plotly_chart(fig_pie, use_container_width=True)
    with col2:
        st.subheader("Client Type Breakdown")
        if 'client_type' in filtered_df.columns:
            fig_bar = px.histogram(filtered_df, x='buyer_segment', color='client_type', barmode='group')
            st.plotly_chart(fig_bar, use_container_width=True)

with tab2:
    st.header("Investor Behavior Dashboard")
    col1, col2 = st.columns(2)
    with col1:
        st.subheader("Acquisition Purpose Profile")
        if 'acquisition_purpose' in filtered_df.columns:
            fig_purpose = px.histogram(filtered_df, x='buyer_segment', color='acquisition_purpose', barmode='stack')
            st.plotly_chart(fig_purpose, use_container_width=True)
    with col2:
        st.subheader("Loan Behavior Profile")
        if 'loan_applied' in filtered_df.columns:
            fig_loan = px.histogram(filtered_df, x='buyer_segment', color='loan_applied', barmode='group')
            st.plotly_chart(fig_loan, use_container_width=True)

with tab3:
    st.header("Geographic Buyer Analysis")
    if 'country' in filtered_df.columns:
        fig_geo = px.histogram(filtered_df, x='country', color='buyer_segment', barmode='group', title="Buyer Distribution by Country")
        st.plotly_chart(fig_geo, use_container_width=True)

with tab4:
    st.header("Segment Insights Panel")
    insights = filtered_df.groupby('buyer_segment').size().reset_index(name='Total_Clients')
    st.dataframe(insights, use_container_width=True)
    st.success("Dashboard components successfully loaded.")

2026-08-26 14:49:53.060 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-26 14:49:53.065 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-26 14:49:53.396 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-08-26 14:49:53.401 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-26 14:49:53.405 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-26 14:49:53.428 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-26 14:49:53.438 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn